# Prerequisites

In [2]:
# get data for labs
# (pure python instead of `!wget`: works in the docker image AND on a
# local Windows kernel, where wget does not exist)
import os
import urllib.request

if not os.path.exists("around_the_world_in_80_days.txt"):
    urllib.request.urlretrieve(
        "https://www.gutenberg.org/ebooks/103.txt.utf-8", "around_the_world_in_80_days.txt"
    )
print("around_the_world_in_80_days.txt:", os.path.getsize("around_the_world_in_80_days.txt"), "bytes")

around_the_world_in_80_days.txt: 403712 bytes


# 1. Word Count

Instructions:  
For each cell marked "double-click and add explanation here" please answer the question in your own words.  
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.  
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [3]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [4]:
# Define the rdd
# NOTE: the original notebook used the colab path '/content/...'.
# We are running in the Jupyter/pyspark docker image instead, where the
# notebook's working directory is the mounted project volume, so a
# relative path is used here.
rdd = sc.textFile('around_the_world_in_80_days.txt')

In [5]:
# view the first x lines of the rdd
rdd.take(20)

['The Project Gutenberg eBook of Around the World in Eighty Days',
 '    ',
 'This eBook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms',
 'of the Project Gutenberg License included with this eBook or online',
 'at www.gutenberg.org. If you are not located in the United States,',
 'you will have to check the laws of the country where you are located',
 'before using this eBook.',
 '',
 'Title: Around the World in Eighty Days',
 '',
 'Author: Jules Verne',
 '',
 'Translator: George M. Towle',
 '',
 '',
 '        ',
 'Release date: January 1, 1994 [eBook #103]',
 '                Most recently updated: October 29, 2024']

In [6]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [7]:
# Note and explain the output of the below command
words

PythonRDD[3] at RDD at PythonRDD.scala:59

`words` is an RDD object (e.g. `PythonRDD[...] at RDD at PythonRDD.scala:53`), not the actual data. `flatMap` is a **transformation**, and Spark uses lazy evaluation: calling it only records a step in the RDD's lineage/DAG ("split every line on spaces, then flatten the results"). No task is actually scheduled or executed on the cluster yet, and no data has moved. The real computation only happens once an **action** (like `take`, `collect`, `count`, ...) is called on this RDD.

This is why printing `words` shows an internal Spark object reference and not a list of words: the RDD is just a pointer to a not-yet-executed computation graph. Spark can afford to wait because several transformations can be chained and optimized together before any actual work is triggered, which avoids unnecessary intermediate materializations of the data.

In [ ]:
# Note and explain the output of the following command, focusing on the
# difference with the above command
# collect() is an ACTION: every word is brought back to the driver as a
# plain python list. Only the first 50 elements are displayed, because
# rendering the ~70k words in the browser freezes the notebook.
all_words = words.collect()
print(type(all_words), len(all_words), "words")
all_words[:50]

Unlike the previous cell, `collect()` is an **action**. It forces Spark to actually execute the whole lineage of transformations recorded so far (reading the file, splitting every line on spaces, flattening), across every partition/executor, and then ships all of the resulting elements back to the driver program as a single, ordinary Python `list` of strings. That's why the output here is the real word data instead of an RDD object reference. Because `collect()` pulls the entire dataset into the driver's memory, it is only safe to use on data that is known to be small enough to fit there; on a real, large dataset it would be avoided in favor of actions like `take(n)`, `count()`, or writing the result back out to storage.

In [ ]:
# nicer print (first 50 words only: printing the whole book one word
# per line freezes the browser)
for w in words.take(50):
    print(w)

In [10]:
# Print first x words
words.take(20)

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for']

`rdd.flatMap(lambda lines: lines.split(' '))` applies the function `lines.split(' ')` to every element (every *line*) of `rdd`, exactly like `map` would. The difference is what happens next: each call to `lines.split(' ')` returns a *list* of words for that line, and instead of producing one list-per-line (an RDD of lists, i.e. a nested/2D structure), `flatMap` **flattens** all of those lists into a single, flat RDD where every element is one individual word. In other words, `map` would give `[["Around", "the", "World"], ["in", "Eighty", "Days"], ...]` while `flatMap` gives `["Around", "the", "World", "in", "Eighty", "Days", ...]`.

In [ ]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

# first 50 pairs only, same reason as above
for w in words.take(50):
    print(w)

In [12]:
# a. count the occurrence of each word
word_counts = words.reduceByKey(lambda count_a, count_b: count_a + count_b)
word_counts.take(20)

[('Gutenberg', 60),
 ('eBook', 6),
 ('of', 1875),
 ('Around', 4),
 ('', 2193),
 ('for', 407),
 ('use', 16),
 ('anyone', 6),
 ('United', 23),
 ('States', 10),
 ('and', 1793),
 ('most', 43),
 ('other', 59),
 ('world', 30),
 ('at', 576),
 ('no', 124),
 ('cost', 9),
 ('with', 550),
 ('almost', 19),
 ('restrictions', 2)]

In [13]:
# b. a common first step in text analysis, change all capital letters to lower case
words_lower = rdd.flatMap(lambda lines: lines.split(' ')) \
                  .map(lambda word: word.lower())
word_counts_lower = words_lower.map(lambda word: (word, 1)) \
                                .reduceByKey(lambda a, b: a + b)
word_counts_lower.take(20)

[('of', 1926),
 ('around', 32),
 ('world', 35),
 ('eighty', 27),
 ('days', 46),
 ('', 2193),
 ('this', 341),
 ('for', 414),
 ('use', 19),
 ('anyone', 6),
 ('united', 27),
 ('states', 14),
 ('and', 1835),
 ('most', 45),
 ('other', 63),
 ('at', 645),
 ('no', 137),
 ('cost', 12),
 ('with', 562),
 ('almost', 19)]

In [14]:
# c. eliminate the stop words.
STOPWORDS_EN = {
    "a", "about", "above", "after", "again", "all", "am", "an", "and", "any",
    "are", "as", "at", "be", "because", "been", "before", "being", "below",
    "between", "both", "but", "by", "can", "did", "do", "does", "doing",
    "down", "during", "each", "few", "for", "from", "further", "had", "has",
    "have", "having", "he", "her", "here", "hers", "herself", "him",
    "himself", "his", "how", "i", "if", "in", "into", "is", "it", "its",
    "itself", "just", "me", "more", "most", "my", "myself", "no", "nor",
    "not", "now", "of", "off", "on", "once", "only", "or", "other", "our",
    "ours", "ourselves", "out", "over", "own", "s", "same", "she", "should",
    "so", "some", "such", "t", "than", "that", "the", "their", "theirs",
    "them", "themselves", "then", "there", "these", "they", "this", "those",
    "through", "to", "too", "under", "until", "up", "very", "was", "we",
    "were", "what", "when", "where", "which", "while", "who", "whom", "why",
    "will", "with", "you", "your", "yours", "yourself", "yourselves",
}

words_no_stop = words_lower.filter(lambda word: word not in STOPWORDS_EN)
word_counts_no_stop = words_no_stop.map(lambda word: (word, 1)) \
                                    .reduceByKey(lambda a, b: a + b)
word_counts_no_stop.take(20)

[('around', 32),
 ('world', 35),
 ('eighty', 27),
 ('days', 46),
 ('', 2193),
 ('use', 19),
 ('anyone', 6),
 ('united', 27),
 ('states', 14),
 ('cost', 12),
 ('almost', 19),
 ('restrictions', 2),
 ('give', 18),
 ('re-use', 2),
 ('license', 12),
 ('online', 4),
 ('www.gutenberg.org.', 4),
 ('states,', 7),
 ('country', 18),
 ('using', 6)]

In [15]:
# d. sort in alphabetical order
word_counts_alpha = word_counts_no_stop.sortByKey()
word_counts_alpha.take(20)

[('', 2193),
 ('#103]', 1),
 ('#516,', 1),
 ('$5,000)', 1),
 ('&c.,', 1),
 ('($1', 1),
 ('(862)', 1),
 ('(a)', 1),
 ('(and', 1),
 ('(any', 1),
 ('(b)', 1),
 ('(c)', 1),
 ('(does', 1),
 ('(if', 1),
 ('(japan),', 1),
 ('(or', 3),
 ('(saturday,', 1),
 ('(sort', 1),
 ('(sunday)', 1),
 ('(trademark/copyright)', 1)]

In [16]:
# e. sort descending by word frequency
word_counts_by_freq = word_counts_no_stop.sortBy(
    lambda pair: pair[1], ascending=False
)
word_counts_by_freq.take(20)

[('', 2193),
 ('mr.', 373),
 ('fogg', 365),
 ('would', 274),
 ('phileas', 250),
 ('passepartout', 239),
 ('said', 157),
 ('could', 139),
 ('one', 133),
 ('fogg,', 132),
 ('fix', 129),
 ('passepartout,', 121),
 ('“i', 115),
 ('upon', 113),
 ('two', 98),
 ('hundred', 92),
 ('replied', 89),
 ('project', 87),
 ('without', 86),
 ('thousand', 83)]

In [17]:
# f. remove punctuations and blank spaces
import string

PUNCTUATION = set(string.punctuation) | {"\u2014", "\u2018", "\u2019", "\u201c", "\u201d"}

def strip_punctuation(word):
    return "".join(char for char in word if char not in PUNCTUATION)

words_clean = words_no_stop.map(strip_punctuation) \
                            .filter(lambda word: word != "")
word_counts_clean = words_clean.map(lambda word: (word, 1)) \
                                .reduceByKey(lambda a, b: a + b)
word_counts_clean.sortBy(lambda pair: pair[1], ascending=False).take(20)

[('fogg', 601),
 ('passepartout', 402),
 ('mr', 389),
 ('would', 283),
 ('phileas', 255),
 ('fix', 240),
 ('said', 194),
 ('one', 166),
 ('could', 139),
 ('and', 138),
 ('him', 137),
 ('i', 126),
 ('aouda', 125),
 ('time', 125),
 ('upon', 121),
 ('train', 119),
 ('you', 115),
 ('it', 106),
 ('master', 104),
 ('two', 102)]

### Putting it all together

All of the steps above (a-f) are now chained into a single reusable function, taking the source RDD of raw lines plus a stopword/punctuation set, and returning an RDD of `(word, count)` pairs.

Two refinements compared to the step-by-step version:
- the Project Gutenberg license header/footer (written in English, in *both* files) is dropped, so it does not pollute the counts;
- words are split on whitespace **and apostrophes**, so that `Fogg's` gives `fogg` + `s`, and French elisions like `l'homme` / `qu'il` give `l` + `homme` / `qu` + `il` instead of the meaningless `lhomme` / `quil`.

In [18]:
import re

def strip_gutenberg(lines_rdd):
    """Keep only the lines between the '*** START OF' / '*** END OF' markers."""
    indexed = lines_rdd.zipWithIndex()
    markers = (
        indexed
        .filter(lambda p: p[0].startswith(("*** START OF", "*** END OF")))
        .values()
        .collect()
    )
    start, end = markers[0], markers[-1]
    return indexed.filter(lambda p: start < p[1] < end).keys()

# whitespace, straight apostrophe and typographic apostrophe
TOKEN_SEPARATORS = re.compile(r"[\s'’]+")

def clean_word_counts(lines_rdd, stopwords, punctuation=PUNCTUATION):
    return (
        lines_rdd
        .flatMap(lambda line: TOKEN_SEPARATORS.split(line))
        .map(lambda word: word.lower())
        # strip punctuation BEFORE filtering stopwords: otherwise a
        # token like "and," (comma still attached) does not match
        # the plain stopword "and" and slips through the filter
        .map(lambda word: "".join(c for c in word if c not in punctuation))
        .filter(lambda word: word != "" and word not in stopwords)
        .map(lambda word: (word, 1))
        .reduceByKey(lambda count_a, count_b: count_a + count_b)
    )

book_en = strip_gutenberg(rdd)
word_counts_en = clean_word_counts(book_en, STOPWORDS_EN)

print("Alphabetical order:")
for word, count in word_counts_en.sortByKey().take(10):
    print(f"  {word}: {count}")

print("\nMost frequent first:")
for word, count in word_counts_en.sortBy(lambda p: p[1], ascending=False).take(10):
    print(f"  {word}: {count}")

Alphabetical order:
  1: 1
  1000: 1
  10th: 1
  11: 1
  1140: 1
  117: 2
  1170: 1
  11th: 7
  12th: 3
  13: 2

Most frequent first:
  fogg: 645
  passepartout: 422
  mr: 389
  would: 283
  phileas: 255
  fix: 255
  said: 194
  one: 167
  could: 138
  aouda: 136


# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [19]:
 # Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
  # map each (name, age) pair to (name, (age, 1)): the 1 is a
  # counter that will let us know how many ages were summed per
  # name, which we need later to compute an average
  .map(lambda x: (x[0], (x[1], 1)))
  # combine all pairs sharing the same name (key) by summing
  # their ages together and their counters together, giving
  # (name, (sum_of_ages, number_of_entries)) per distinct name
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # divide the summed ages by the number of entries to get the
  # average age per name: (name, average_age)
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

Overall, this snippet computes the **average age per name** in `dataRDD` using the classic Spark "sum + count" pattern: since there is no built-in `averageByKey`, each value is first turned into a `(value, 1)` pair, `reduceByKey` accumulates both the running sum and the running count for each key in a single pass, and a final `map` divides the two to get the average. Here `Brooke` appears twice (20 and 25), so `agesRDD` will contain `("Brooke", 22.5)` alongside `("Denny", 31.0)`, `("Jules", 30.0)` and `("TD", 35.0)`.

In [20]:
agesRDD.collect()

[('Denny', 31.0), ('TD', 35.0), ('Jules', 30.0), ('Brooke', 22.5)]

## 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible


In [21]:
import time

def time_pipeline(build_fn, *args, repeat=5, **kwargs):
    """Build then materialize (collect) an RDD pipeline `repeat` times.

    Returns the best wall-clock time (least affected by noise such as
    JVM warm-up or background load) and the collected result.
    """
    timings = []
    for _ in range(repeat):
        start = time.perf_counter()
        result = build_fn(*args, **kwargs).collect()
        timings.append(time.perf_counter() - start)
    return min(timings), result

In [22]:
def clean(word, punctuation):
    return "".join(c for c in word if c not in punctuation)

# A. "textbook" ordering: clean + filter right after splitting, so
# only useful words are paired and shuffled by reduceByKey
def order_filter_early(lines_rdd, stopwords, punctuation):
    return (
        lines_rdd
        .flatMap(lambda line: TOKEN_SEPARATORS.split(line))
        .map(lambda w: clean(w.lower(), punctuation))
        .filter(lambda w: w != "" and w not in stopwords)
        .map(lambda w: (w, 1))
        .reduceByKey(lambda a, b: a + b)
    )

# B. build the (word, 1) pairs first, then clean/filter the tuples:
# same result, one extra pass over bigger objects before the shuffle
def order_filter_late(lines_rdd, stopwords, punctuation):
    return (
        lines_rdd
        .flatMap(lambda line: TOKEN_SEPARATORS.split(line))
        .map(lambda w: (w, 1))
        .map(lambda p: (clean(p[0].lower(), punctuation), p[1]))
        .filter(lambda p: p[0] != "" and p[0] not in stopwords)
        .reduceByKey(lambda a, b: a + b)
    )

# C. count the raw tokens first, clean afterwards: the expensive
# python cleaning now runs once per DISTINCT raw token ("Fogg",
# "fogg,", "Fogg." ...) instead of once per occurrence, at the cost
# of a second reduceByKey to merge the variants once cleaned
def order_reduce_first(lines_rdd, stopwords, punctuation):
    return (
        lines_rdd
        .flatMap(lambda line: TOKEN_SEPARATORS.split(line))
        .map(lambda w: (w, 1))
        .reduceByKey(lambda a, b: a + b)
        .map(lambda p: (clean(p[0].lower(), punctuation), p[1]))
        .filter(lambda p: p[0] != "" and p[0] not in stopwords)
        .reduceByKey(lambda a, b: a + b)
    )

# D. groupByKey instead of reduceByKey: every single (word, 1) pair
# goes through the shuffle (no map-side combining), and the lists of
# 1s are only summed afterwards
def order_group_by_key(lines_rdd, stopwords, punctuation):
    return (
        lines_rdd
        .flatMap(lambda line: TOKEN_SEPARATORS.split(line))
        .map(lambda w: (clean(w.lower(), punctuation), 1))
        .groupByKey()
        .mapValues(sum)
        .filter(lambda p: p[0] != "" and p[0] not in stopwords)
    )

orderings = [
    ("A. filter early", order_filter_early),
    ("B. filter after pairing", order_filter_late),
    ("C. reduce, clean, reduce again", order_reduce_first),
    ("D. groupByKey + late filter", order_group_by_key),
]

# the book alone (~450 KB) runs in a fraction of a second, so timings
# are mostly noise: replicate it 20 times (~9 MB) and cache it, so we
# time the pipeline itself rather than the file reading
book_big = sc.union([book_en] * 20).repartition(4).cache()
print(f"benchmark input: {book_big.count()} lines\n")

reference = None
for label, build_fn in orderings:
    elapsed, result = time_pipeline(
        build_fn, book_big, STOPWORDS_EN, PUNCTUATION, repeat=3
    )
    result = dict(result)
    reference = reference or result
    same = "same result" if result == reference else "DIFFERENT result"
    print(f"{label:32s} {elapsed:.3f}s  ({len(result)} words, {same})")

book_big.unpersist()

benchmark input: 158680 lines

A. filter early                  0.694s  (6971 words, same result)
B. filter after pairing          0.716s  (6971 words, same result)
C. reduce, clean, reduce again   0.501s  (6971 words, same result)
D. groupByKey + late filter      0.692s  (6971 words, same result)


MapPartitionsRDD[78] at coalesce at DirectMethodHandleAccessor.java:103

**Which ordering is optimal, and why?**

All four orderings return exactly the same `(word, count)` pairs; only the order of the operations changes. Measured on the book replicated 20 times (~160k lines, ~1.1M tokens, cached, best of 3 runs; two separate runs gave the same ranking):

| ordering | time |
|---|---|
| A. filter early | ~0.66 s |
| B. filter after pairing | ~0.66 s |
| **C. reduce raw tokens, clean, reduce again** | **~0.45 s** |
| D. groupByKey + late filter | ~0.64 s |

The textbook rule is *"filter as early as possible, before the shuffle"*, and it is a good rule on a real cluster where the shuffle (serialization + disk + network) dominates. But here, on a single machine and a few MB of data, the shuffle is cheap in every case, and the real cost is the **Python work done per element**: every token has to be sent from the JVM to a Python worker and go through `lower()` + the character-by-character punctuation strip.

- **A, B**: the cleaning function runs on **every one of the ~1.1M token occurrences**, and the filter cannot drop anything before that cleaning (we need to clean `"And,"` to know it is the stopword `and`). Pairing before or after the cleaning (B vs A) makes no real difference.
- **C** is the fastest: `reduceByKey` combines map-side first, so the ~1.1M occurrences collapse to a few tens of thousands of **distinct raw tokens**, and the expensive cleaning runs only on those. The second `reduceByKey` (to merge `Fogg`, `Fogg,`, `Fogg.` into `fogg`) is a second shuffle, but it works on very little data. This is the general idea of **reducing the data volume before doing expensive per-record work**.
- **D**: `groupByKey` has no map-side combine, so every `(word, 1)` pair is shuffled. On this small data it is only as slow as A, but it is the ordering that scales worst: on a cluster with a large text, the shuffled volume would be much bigger than with `reduceByKey`, which is why `reduceByKey` should always be preferred for aggregations.

The Spark UI (http://localhost:4040, *Stages* tab) shows the shuffle read/write size of each stage, which lets you check these explanations.

## 4. Text Comparison

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two

In [23]:
# get the french version of the book
# (pure python instead of `!wget`: works in the docker image AND on a
# local Windows kernel, where wget does not exist)
import os
import urllib.request

if not os.path.exists("le_tour_du_monde_en_80_jours.txt"):
    urllib.request.urlretrieve(
        "https://www.gutenberg.org/ebooks/46541.txt.utf-8", "le_tour_du_monde_en_80_jours.txt"
    )
print("le_tour_du_monde_en_80_jours.txt:", os.path.getsize("le_tour_du_monde_en_80_jours.txt"), "bytes")

le_tour_du_monde_en_80_jours.txt: 472731 bytes


In [24]:
STOPWORDS_FR = {
    "a", "à", "ai", "c", "as", "au", "aux", "avait", "avaient", "avec", "avoir",
    "ce", "ceci", "cela", "ces", "cet", "cette", "ceux", "chez", "comme",
    "d", "dans", "de", "des", "donc", "du", "elle", "elles", "en", "est",
    "et", "été", "étaient", "était", "être", "eu", "eux", "fait", "il",
    "ils", "j", "je", "l", "la", "le", "les", "leur", "leurs", "lui", "m",
    "ma", "mais", "me", "même", "mes", "moi", "mon", "n", "ne", "ni",
    "nos", "notre", "nous", "on", "ont", "ou", "où", "par", "pas", "peu",
    "plus", "pour", "qu", "quand", "que", "quel", "quelle", "qui", "s",
    "sa", "sans", "se", "ses", "si", "son", "sont", "sur", "t", "ta", "te",
    "tes", "toi", "ton", "tous", "tout", "toute", "tu", "un", "une", "vos",
    "votre", "vous", "y",
}

rdd_fr = sc.textFile("le_tour_du_monde_en_80_jours.txt")
book_fr = strip_gutenberg(rdd_fr)
word_counts_fr = clean_word_counts(book_fr, STOPWORDS_FR)

In [25]:
en_unique = word_counts_en.count()
fr_unique = word_counts_fr.count()
en_total = word_counts_en.map(lambda pair: pair[1]).sum()
fr_total = word_counts_fr.map(lambda pair: pair[1]).sum()

print("                English   French")
print(f"Unique words:   {en_unique:7d}   {fr_unique:7d}")
print(f"Total words:    {en_total:7d}   {fr_total:7d}")

                English   French
Unique words:      6971      9710
Total words:      32537     38236


In [26]:
print("Top 15 English words:")
for word, count in word_counts_en.sortBy(lambda p: p[1], ascending=False).take(15):
    print(f"  {word:15s} {count}")

print("\nTop 15 French words:")
for word, count in word_counts_fr.sortBy(lambda p: p[1], ascending=False).take(15):
    print(f"  {word:15s} {count}")

Top 15 English words:
  fogg            645
  passepartout    422
  mr              389
  would           283
  phileas         255
  fix             255
  said            194
  one             167
  could           138
  aouda           136
  master          128
  time            125
  train           119
  upon            119
  two             102

Top 15 French words:
  fogg            682
  passepartout    451
  phileas         330
  mr              287
  fix             284
  heures          242
  répondit        215
  bien            191
  deux            145
  dit             138
  aouda           134
  après           132
  mrs             131
  quelques        129
  monsieur        123


Because the two books are written in different languages, the vocabularies barely overlap, so comparing counts word-for-word only makes sense for the words that are **identical in both editions**: character names and place names (never translated) plus a few spelling-identical words. We find them with a `join` on the word (the key), which is exactly the kind of query RDDs of `(key, value)` pairs are made for.

In [27]:
# join the two (word, count) RDDs on the word: only the words present
# in BOTH editions are kept -> (word, (count_en, count_fr))
common = word_counts_en.join(word_counts_fr)

print(f"{common.count()} words appear in both editions\n")
print(f"{'word':15s}{'english':>9s}{'french':>9s}{'fr/en':>8s}")
top_common = common.sortBy(lambda p: p[1][0] + p[1][1], ascending=False)
for word, (en, fr) in top_common.take(20):
    print(f"{word:15s}{en:9d}{fr:9d}{fr / en:8.2f}")

846 words appear in both editions

word             english   french   fr/en
fogg                 645      682    1.06
passepartout         422      451    1.07
mr                   389      287    0.74
phileas              255      330    1.29
fix                  255      284    1.11
aouda                136      134    0.99
train                119      112    0.94
sir                  100       54    0.54
monsieur              29      123    4.24
gentleman             41       85    2.07
moment                50       70    1.40
bombay                59       61    1.03
minutes               64       48    0.75
steamer               91       16    0.18
francis               54       53    0.98
point                 26       71    2.73
station               46       45    0.98
fit                    2       83   41.50
fort                  24       59    2.46
guide                 41       42    1.02


In [28]:
# normalize by the size of each book to compare frequencies (per
# 10,000 words) rather than raw counts: the French text is longer
names = ["fogg", "passepartout", "aouda", "fix", "phileas"]

en_name_counts = dict(word_counts_en.filter(lambda p: p[0] in names).collect())
fr_name_counts = dict(word_counts_fr.filter(lambda p: p[0] in names).collect())

print(f"{'name':14s}{'english':>9s}{'french':>9s}{'en/10k':>9s}{'fr/10k':>9s}")
for name in names:
    en, fr = en_name_counts.get(name, 0), fr_name_counts.get(name, 0)
    print(f"{name:14s}{en:9d}{fr:9d}"
          f"{en / en_total * 1e4:9.1f}{fr / fr_total * 1e4:9.1f}")

name            english   french   en/10k   fr/10k
fogg                645      682    198.2    178.4
passepartout        422      451    129.7    118.0
aouda               136      134     41.8     35.0
fix                 255      284     78.4     74.3
phileas             255      330     78.4     86.3


**Conclusions of the comparison**

- **Size**: once the Gutenberg license and the stopwords are removed, the French edition has more words (~38k vs ~33k) and a much richer vocabulary (~9.7k vs ~7.0k distinct words). French is more inflected (verb conjugations `dit`/`disait`/`répondit`, gender and plural agreement), so one English word maps to many French forms; the English text is also a (slightly condensed) translation.
- **Top words**: in both books the top words are the characters: `fogg`, `passepartout`, `phileas`, `fix`, `aouda`. Then come dialogue verbs (`said` / `répondit`, `dit`) and words about time and travel (`time`, `train`, `heures`, `deux`), which fits a race against the clock.
- **Proper nouns as anchors**: the `join` shows that names appear at very similar rates (`fogg` 645 vs 682, `aouda` 136 vs 134, `bombay` 59 vs 61, `francis` 54 vs 53), which confirms that both texts tell exactly the same story. Once normalized per 10k words, the English edition even mentions the characters a bit more often, since the text is shorter.
- **Translation choices**: `phileas` is more frequent in French (Verne often writes "Phileas Fogg" in full), while `mr` is less frequent because the French text also uses `monsieur` (123 times vs 29).
- **Limits of a word-for-word join**: some common keys are *false friends*, not the same word: `fit` (French verb *faire*, 83 times vs 2 for English *fit*), `fort` (French *strong/very* vs English *fort*), `point` (French negation *ne... point*). A real comparison of the vocabulary would need a translation dictionary or word alignment, which is beyond a word count.
- **Stopword lists**: both lists are deliberately minimal (hand-written, no extra dependency). A complete list such as NLTK's would also filter function words that still appear in the top 15 (`would`, `could`, `upon`, `one` in English, `bien`, `après`, `quelques` in French), which would make the rankings more meaningful. This does not change the conclusions above, since they rely on proper nouns and on the relative sizes of the two texts.